# 1. Ingesta y curaduría de datos experimentales
**Objetivo:** convertir una fuente real en una tabla trazable, conservando los originales.
Una fila debe representar un caso/espécimen ensayado. No crear filas independientes por cada
rotura o ciclo del mismo ensayo. Esta etapa no imputa, escala ni entrena modelos.

Entradas: fuente autorizada en `data/raw/` y esquema revisado. Salidas: versiones en
`data/interim/` y `data/processed/`, hashes, reportes y ordinales de procedencia.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'ingesta.py').is_file()
             and (p / 'config' / 'data_schema.yml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abra Jupyter desde el repositorio o notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import pandas as pd
import yaml
from src.ingesta import ingest
from src.preprocesamiento import curate_file

RAW_FILE = None       # Ruta relativa a ROOT de un CSV/XLSX real
INTERIM_FILE = None   # Ruta indicada por una ingesta anterior
RUN_INGESTION = False
RUN_CURATION = False
schema = yaml.safe_load((ROOT / 'config' / 'data_schema.yml').read_text(encoding='utf-8'))
schema


## Procedencia y definición física
Registrar institución/proveedor, versión, archivo, hoja, unidades y relación entre identificador,
espécimen y configuración. Conciliar la tabla general con las fichas individuales antes de
aceptar discrepancias. El target es la fuerza al primer evento localizado de rotura, no el máximo
de toda la curva ni una caída debida a descarga programada.

En `data_schema.yml`, declarar nombres reales, unidades y marcadores de ausencia por columna.
Conservar duplicados hasta distinguir copias de registro de réplicas experimentales.


In [ ]:
if RUN_INGESTION:
    if RAW_FILE is None:
        raise ValueError('Indique un archivo original real.')
    _, manifest = ingest(Path(RAW_FILE))
    INTERIM_FILE = manifest['interim_file']
    display(manifest)
else:
    print('Ingesta preparada; no se ha ejecutado.')


In [ ]:
quality = None
if RUN_CURATION:
    if INTERIM_FILE is None:
        raise ValueError('Indique una ingesta real versionada.')
    quality = curate_file(Path(INTERIM_FILE), ROOT / 'config' / 'data_schema.yml')
    display(pd.DataFrame(quality['excluded_records']))
    display(pd.DataFrame({'antes': quality['before']['missing_by_column'],
                          'después': quality['after']['missing_by_column']}))
    print('Tabla:', quality['output_file'])
else:
    print('Curaduría preparada; requiere completar y verificar el esquema.')


## Decisiones que deben quedar justificadas
- Definición y unidad del target; correspondencia con el primer evento de falla.
- Registros excluidos y motivo; no imputar la respuesta.
- Duplicados, identificadores repetidos, réplicas y columnas para agrupar observaciones.
- Anomalías físicas, constantes, dependencias geométricas y variables redundantes.
- Disponibilidad de cada predictor antes del ensayo.

**Criterio de salida:** tabla curada y reporte revisados. Continuar con `EDA_basico.ipynb`.
